# 04 Analysis

Segunda fase del flujo. Este notebook parte desde las salidas curadas de **procesamiento** y no recalcula normalizacion, `ROI_status`, `phase`, `trend` ni las metricas `low/mid/high` ya generadas.

Entrada canonica de esta fase:

```text
data/Proc_data/batch_analysis/processing_active.csv
```

Primer objetivo: filtrar por `trend` y visualizar una curva fina de `NormSignal` por temperatura.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Parametros

Edita estos valores para explorar otros subconjuntos sin cambiar el procesamiento original.

In [ ]:
cwd = Path.cwd()
project_root = cwd.parent if cwd.name == "notebooks" else cwd

base_dir = project_root / "data" / "Proc_data"
processing_path = base_dir / "batch_analysis" / "processing_active.csv"

trend_filter = ["increase"]
mutant_filter = ["m27", "m36", "m45", "m65"]  # None conserva todos; ejemplo: ["m27", "m36"]
phase_filter = "cooling"
temp_range = (25,40)  # None conserva todo el rango; ejemplo: (20, 40)
bin_width = 1.0
value_col = "NormSignal"

# Usa [] para una curva global, o columnas como ["genotype_meta"] o ["genotype_meta", "sample"].
curve_group_cols = ["genotype_meta"]

## Cargar salida de procesamiento

Esta fase solo lee `processing_active.csv`. Si falta alguna columna minima, el notebook se detiene para evitar analisis ambiguos.

In [9]:
required_cols = {
    "sample",
    "genotype_meta",
    "ROI",
    "trend",
    "phase",
    "temp_mean",
    value_col,
}

if not processing_path.exists():
    raise FileNotFoundError(f"No existe la entrada de analysis: {processing_path}")

processing_active = pd.read_csv(processing_path, low_memory=False)
missing_cols = required_cols - set(processing_active.columns)
if missing_cols:
    raise ValueError(f"Faltan columnas requeridas en processing_active.csv: {sorted(missing_cols)}")

print(f"Fuente: {processing_path}")
print(f"processing_active: {processing_active.shape}")
print(f"ROIs: {processing_active['ROI'].nunique()}")
print(f"Muestras: {processing_active['sample'].nunique()}")
print(f"Genotipos: {processing_active['genotype_meta'].nunique()}")

Fuente: /Users/gfernandezv/Documents/envs/Images_TTL_temp/data/Proc_data/batch_analysis/processing_active.csv
processing_active: (139165, 38)
ROIs: 391
Muestras: 8
Genotipos: 5


## Filtrar por fase y trend

El filtro respeta las etiquetas heredadas desde procesamiento; no las recalcula.

In [10]:
analysis_df = processing_active.copy()

if phase_filter is not None:
    analysis_df = analysis_df[analysis_df["phase"].astype(str).eq(str(phase_filter))].copy()

if trend_filter:
    selected_trends = {str(trend) for trend in trend_filter}
    analysis_df = analysis_df[analysis_df["trend"].astype(str).isin(selected_trends)].copy()

if mutant_filter:
    selected_mutants = {str(mutant) for mutant in mutant_filter}
    analysis_df = analysis_df[analysis_df["genotype_meta"].astype(str).isin(selected_mutants)].copy()

analysis_df["temp_mean"] = pd.to_numeric(analysis_df["temp_mean"], errors="coerce")
analysis_df[value_col] = pd.to_numeric(analysis_df[value_col], errors="coerce")
analysis_df = analysis_df.dropna(subset=["temp_mean", value_col, "ROI"]).copy()

if temp_range is not None:
    temp_min, temp_max = temp_range
    analysis_df = analysis_df[
        analysis_df["temp_mean"].between(temp_min, temp_max, inclusive="both")
    ].copy()

if analysis_df.empty:
    raise ValueError("El filtro no dejo filas para analizar. Revisa phase_filter, trend_filter, mutant_filter y temp_range.")

print(f"phase_filter: {phase_filter}")
print(f"trend_filter: {trend_filter}")
print(f"mutant_filter: {mutant_filter}")
print(f"temp_range: {temp_range}")
print(f"Filas filtradas: {analysis_df.shape}")
print(f"ROIs filtradas: {analysis_df['ROI'].nunique()}")
print(f"Muestras filtradas: {analysis_df['sample'].nunique()}")
print(f"Genotipos filtrados: {analysis_df['genotype_meta'].nunique()}")

display(
    analysis_df[["sample", "genotype_meta", "ROI", "trend", "phase", "temp_mean", value_col]]
    .head()
)

phase_filter: cooling
trend_filter: ['increase']
mutant_filter: ['m27', 'm36', 'm45', '65']
temp_range: (25, 40)
Filas filtradas: (34677, 38)
ROIs filtradas: 360
Muestras filtradas: 4
Genotipos filtrados: 3


,sample,genotype_meta,ROI,trend,phase,temp_mean,NormSignal
33,sample_08,m36,ROI1,increase,cooling,39.549322,0.061344
34,sample_08,m36,ROI1,increase,cooling,39.215939,0.045983
35,sample_08,m36,ROI1,increase,cooling,38.574827,0.046308
36,sample_08,m36,ROI1,increase,cooling,38.153295,0.044764
37,sample_08,m36,ROI1,increase,cooling,37.794258,0.032248


## Curva fina por temperatura

Primero se promedia cada ROI dentro de cada bin de temperatura. Luego se resume la distribucion de ROIs por bin para graficar la curva.

In [11]:
temp_start = np.floor(analysis_df["temp_mean"].min() / bin_width) * bin_width
temp_stop = np.ceil(analysis_df["temp_mean"].max() / bin_width) * bin_width + bin_width
temp_bins = np.arange(temp_start, temp_stop + bin_width / 10, bin_width)

binned_df = analysis_df.copy()
binned_df["temp_bin"] = pd.cut(
    binned_df["temp_mean"],
    bins=temp_bins,
    include_lowest=True,
    right=False,
)

roi_group_cols = ["genotype_meta", "sample", "ROI", "temp_bin"]
roi_bin_summary = (
    binned_df.dropna(subset=["temp_bin"])
    .groupby(roi_group_cols, observed=True, dropna=False)
    .agg(
        temp_center=("temp_mean", "mean"),
        signal_mean=(value_col, "mean"),
        n_points=(value_col, "size"),
    )
    .reset_index()
)

summary_group_cols = [col for col in curve_group_cols if col in roi_bin_summary.columns]
curve_summary = (
    roi_bin_summary
    .groupby(summary_group_cols + ["temp_bin"], observed=True, dropna=False)
    .agg(
        temp_center=("temp_center", "mean"),
        mean_signal=("signal_mean", "mean"),
        sd_signal=("signal_mean", "std"),
        n_roi=("ROI", "nunique"),
    )
    .reset_index()
)
curve_summary["sem_signal"] = curve_summary["sd_signal"] / np.sqrt(curve_summary["n_roi"])

if summary_group_cols:
    phenotype_counts = (
        analysis_df
        .groupby(summary_group_cols, dropna=False)
        .agg(
            n_roi=("ROI", "nunique"),
            n_samples=("sample", "nunique"),
            n_rows=(value_col, "size"),
        )
        .reset_index()
    )
else:
    phenotype_counts = pd.DataFrame({
        "phenotype": ["all"],
        "n_roi": [analysis_df["ROI"].nunique()],
        "n_samples": [analysis_df["sample"].nunique()],
        "n_rows": [len(analysis_df)],
    })

print(f"bin_width: {bin_width} °C")
print(f"ROI x bin: {roi_bin_summary.shape}")
print(f"Curva resumida: {curve_summary.shape}")
print("n por fenotipo presentado en el grafico:")
display(phenotype_counts)

display(curve_summary.head())

bin_width: 1.0 °C
ROI x bin: (10695, 7)
Curva resumida: (45, 7)
n por fenotipo presentado en el grafico:


,genotype_meta,n_roi,n_samples,n_rows
0,m27,286,2,15235
1,m36,276,1,15732
2,m45,70,1,3710


,genotype_meta,temp_bin,temp_center,mean_signal,sd_signal,n_roi,sem_signal
0,m27,"[25.0, 26.0)",25.448823,-0.000330,0.005874,286,0.000347
1,m27,"[26.0, 27.0)",26.555745,-0.000589,0.011497,286,0.000680
2,m27,"[27.0, 28.0)",27.487395,0.001349,0.019570,286,0.001157
3,m27,"[28.0, 29.0)",28.517974,0.005175,0.028962,286,0.001713
4,m27,"[29.0, 30.0)",29.547844,0.012633,0.038118,286,0.002254


In [ ]:
group_col = "genotype_meta"
genotypes = sorted(analysis_df[group_col].dropna().astype(str).unique())

ncols = 2
nrows = int(np.ceil(len(genotypes) / ncols)) if genotypes else 1

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows), sharex=True, sharey=True, squeeze=False)
axes_flat = axes.flatten()

for ax, genotype in zip(axes_flat, genotypes):
    roi_sub = roi_bin_summary[roi_bin_summary[group_col].astype(str).eq(genotype)]
    for _, roi_curve in roi_sub.groupby(["sample", "ROI"], observed=True):
        roi_curve = roi_curve.sort_values("temp_center")
        ax.plot(roi_curve["temp_center"], roi_curve["signal_mean"], color="grey", alpha=0.25, linewidth=0.8)

    mean_sub = curve_summary[curve_summary[group_col].astype(str).eq(genotype)].sort_values("temp_center")
    sem = mean_sub["sem_signal"].fillna(0)
    ax.plot(mean_sub["temp_center"], mean_sub["mean_signal"], color="crimson", linewidth=2.2, marker="o", markersize=4, label="promedio")
    ax.fill_between(mean_sub["temp_center"], mean_sub["mean_signal"] - sem, mean_sub["mean_signal"] + sem, color="crimson", alpha=0.2)

    n_roi = roi_sub[["sample", "ROI"]].drop_duplicates().shape[0]
    ax.axhline(0, color="black", linestyle="--", linewidth=1, alpha=0.4)
    ax.set_title(f"{genotype} (n_roi={n_roi})")
    ax.set_xlabel("Temperature (°C)")
    ax.legend(loc="upper left", fontsize=8)

for ax in axes_flat[: len(genotypes)]:
    if ax.get_subplotspec().is_first_col():
        ax.set_ylabel(value_col)
for ax in axes_flat[len(genotypes):]:
    ax.axis("off")

fig.suptitle(f"Curvas por ROI + promedio | trend={trend_filter} | phase={phase_filter} | temp={temp_range}")
fig.tight_layout()
plt.show()

## Fase 2: Relacion Arrhenius (ln Amp vs 1/T)

Reutiliza el subconjunto ya filtrado en la fase 1 (`analysis_df`: `phase_filter`, `trend_filter`, `mutant_filter`, `temp_range`) para mantener consistencia entre ambas fases.

Para cada ROI se usa la amplitud pico normalizada `(Imax - I_baseline) / I_baseline` (equivalente al maximo de `NormSignal`) y la temperatura a la que ocurre ese pico (`T_at_Imax`). Se calcula `1/T (K)` y `ln(Amp)`.

Solo se conservan ROIs con `Amp > 0`, ya que `ln` no esta definido para valores <= 0.

In [ ]:
roi_id_cols = ["sample", "genotype_meta", "ROI"]

roi_amp_df = (
    analysis_df[roi_id_cols + ["I_baseline", "Imax", "T_at_Imax"]]
    .drop_duplicates(subset=roi_id_cols)
    .copy()
)

roi_amp_df["amp"] = (roi_amp_df["Imax"] - roi_amp_df["I_baseline"]) / roi_amp_df["I_baseline"]

n_before = len(roi_amp_df)
roi_amp_df = roi_amp_df[roi_amp_df["amp"] > 0].copy()
n_dropped = n_before - len(roi_amp_df)

roi_amp_df["T_K"] = roi_amp_df["T_at_Imax"] + 273.15
roi_amp_df["inv_T"] = 1.0 / roi_amp_df["T_K"]
roi_amp_df["ln_amp"] = np.log(roi_amp_df["amp"])

print(f"Filtros heredados de fase 1 -> phase: {phase_filter} | trend: {trend_filter} | mutant: {mutant_filter} | temp_range: {temp_range}")
print(f"ROIs con amp <= 0 descartados: {n_dropped} / {n_before}")
print(f"ROIs usados: {len(roi_amp_df)}")
display(roi_amp_df[["sample", "genotype_meta", "ROI", "amp", "T_at_Imax", "inv_T", "ln_amp"]].head())

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for genotype, sub in roi_amp_df.groupby("genotype_meta", dropna=False):
    sub = sub.sort_values("inv_T")
    x = sub["inv_T"].to_numpy(dtype=float)
    y = sub["ln_amp"].to_numpy(dtype=float)
    ax.scatter(x, y, s=18, alpha=0.7, label=f"{genotype} (n={len(sub)})")

    if len(sub) >= 2 and pd.Series(x).nunique() >= 2:
        slope, intercept = np.polyfit(x, y, deg=1)
        x_fit = np.linspace(x.min(), x.max(), 50)
        ax.plot(x_fit, slope * x_fit + intercept, linewidth=1.5, alpha=0.8)

ax.set_xlabel("1 / T (K$^{-1}$)")
ax.set_ylabel("ln(Amp normalizado)")
ax.set_title(f"Arrhenius plot | trend={trend_filter} | mutant={mutant_filter} | phase={phase_filter}")
ax.legend(title="genotype_meta", bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()
plt.show()